# YOLOv8n fine-tuning
        **vision › detection** · notebook `yolov8n_finetune`

        **Kapan dipakai:** Alur paling praktis untuk lomba deteksi: data format YOLO, latih beberapa menit di GPU.

        **Jebakan umum:** Butuh konversi anotasi ke format YOLO (dibuat otomatis di notebook). Lisensi AGPL-3.0.

> ⚠️ Notebook ini berat (butuh torch / bobot / GPU). Lihat sel pertama.

**Data demo:** Penn-Fudan (170 gambar pejalan kaki, kotak + mask); 120 latih / 50 uji tetap. Metrik: **AP50** (kotak benar bila IoU ≥ 0.5). Bobot dibaca dari `pretrained/` (offline).

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
MODEL_KEY = "yolov8n_finetune"
OUT = out_dir("vision", "detection", MODEL_KEY)
SEED = 42; set_seed(SEED)
SCORE_THR = 0.05                 # ambang skor deteksi (rendah untuk AP; naikkan untuk hasil bersih)
EPOCHS = 15

## 2. Data: gambar + kotak pejalan kaki (dari mask instance)
120 gambar pertama latih, 50 terakhir uji (tetap).

In [ ]:
from superdb.vision_det import load_pennfudan, split_idx, ap50, box_iou
D = load_pennfudan(); itr, ite = split_idx(len(D), 50)
imgs = [d["img"] for d in D]; boxes = [d["boxes"] for d in D]
print(len(D), "gambar | rata-rata orang per gambar:", round(float(np.mean([len(b) for b in boxes])), 2))
import matplotlib.patches as mp
fig, ax = plt.subplots(1, 3, figsize=(10, 3.5))
for a, i in zip(ax, (0, 1, 2)):
    a.imshow(imgs[i]); [a.add_patch(mp.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, color="lime", lw=2)) for b in boxes[i]]; a.axis("off")
plt.tight_layout(); plt.show()

## 3. YOLOv8n (ultralytics) dengan bobot lokal `pretrained/yolov8n/yolov8n.pt`

In [ ]:
from pathlib import Path
import torch
from ultralytics import YOLO
from superdb.paths import ROOT as R
WEIGHTS = R / "pretrained" / "yolov8n" / "yolov8n.pt"
assert WEIGHTS.exists(), "bobot belum ada: python tools/download_pretrained_vision.py yolov8n"
dev = 0 if torch.cuda.is_available() else "cpu"
def yolo_predict(m, ims, conf=SCORE_THR, cls=0):
    out = []
    for r in m.predict(source=[x[:, :, ::-1] for x in ims], conf=conf, device=dev, verbose=False, imgsz=416):    # ultralytics menganggap BGR
        k = r.boxes.cls.cpu().numpy() == cls; out.append((r.boxes.xyxy.cpu().numpy()[k], r.boxes.conf.cpu().numpy()[k]))
    return out

## 4. Siapkan dataset format YOLO
Struktur: `images/{train,val}`, `labels/{train,val}` dengan baris `kelas cx cy w h` ternormalisasi, plus `data.yaml`. Dibuat otomatis dari kotak PennFudan.

In [ ]:
import shutil
from PIL import Image
YD = R / "outputs" / "_cache" / "pennfudan_yolo"
if YD.exists(): shutil.rmtree(YD)
for split, idxs in (("train", itr), ("val", ite)):
    (YD / "images" / split).mkdir(parents=True); (YD / "labels" / split).mkdir(parents=True)
    for i in idxs:
        h, w = imgs[i].shape[:2]; Image.fromarray(imgs[i]).save(YD / "images" / split / f"{i:03d}.png")
        (YD / "labels" / split / f"{i:03d}.txt").write_text("\n".join(f"0 {(b[0] + b[2]) / 2 / w:.6f} {(b[1] + b[3]) / 2 / h:.6f} {(b[2] - b[0]) / w:.6f} {(b[3] - b[1]) / h:.6f}" for b in boxes[i]))
(YD / "data.yaml").write_text(f"path: {YD.as_posix()}\ntrain: images/train\nval: images/val\nnames:\n  0: person\n")
print("dataset YOLO:", len(list((YD / 'images' / 'train').glob('*'))), "latih,", len(list((YD / 'images' / 'val').glob('*'))), "val")

## 5. Latih (fine-tuning dari bobot COCO)
`amp=False` dan `plots=False` agar tidak mengunduh apa pun saat offline.

In [ ]:
base = YOLO(str(WEIGHTS)); ap_base = ap50(yolo_predict(base, [imgs[i] for i in ite]), [boxes[i] for i in ite])["AP50"]
model = YOLO(str(WEIGHTS))
model.train(data=str(YD / "data.yaml"), epochs=EPOCHS, imgsz=416, batch=8, device=dev, workers=0, amp=False, plots=False, verbose=False,
            project=str(OUT / "runs"), name="train", exist_ok=True, seed=SEED, patience=100)
best = YOLO(str(OUT / "runs" / "train" / "weights" / "best.pt")); preds = yolo_predict(best, [imgs[i] for i in ite])
metrics = ap50(preds, [boxes[i] for i in ite]); metrics["AP50_tanpa_finetune(COCO)"] = ap_base; display(show_table(metrics))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 3.5))
import matplotlib.patches as mp
for a, k in zip(ax, (0, 1, 2)):
    i = ite[k]; a.imshow(imgs[i]); a.axis("off")
    [a.add_patch(mp.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, color="lime", lw=2)) for b in boxes[i]]
    b_, s_ = preds[k]; [a.add_patch(mp.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, color="red", lw=1.5, ls="--")) for b, s in zip(b_, s_) if s > 0.4]
plt.tight_layout(); plt.savefig(OUT / "detections.png", dpi=130); plt.show()
shutil.copy(OUT / "runs" / "train" / "weights" / "best.pt", OUT / "model.pt")
save_json({"metrics": metrics}, OUT / "metrics.json"); print("OK. hasil di", OUT)

> **Baca hasilnya dengan jujur:** kelas 'person' sudah ada di COCO, jadi model bawaan sudah hampir sempurna dan fine-tuning hampir tidak menambah skor (bandingkan `AP50` dengan `AP50_tanpa_finetune`). Fine-tuning baru berarti untuk **kelas/domain baru** (mis. jenis produk, objek satelit) yang tidak ada di COCO.

## Langkah berikutnya
- Skor ambang rendah menghasilkan banyak kotak palsu; pilih ambang dari kurva precision-recall pada validation.
- Data lomba: ubah loader (`load_pennfudan`) menjadi pembaca anotasi lomba (COCO JSON/Pascal VOC/YOLO). Evaluasi `ap50` bisa dipakai apa adanya.
- Gambar besar dengan objek kecil: naikkan `imgsz`.